# INV-M365-DN - Standalone Graph Runtime Pack - P3 Prototype Notebook
**Plan reference:** `plan:m365-standalone-graph-runtime-integration-pack:R4`  
**Phase:** `P3` Notebook Proof And Scorecard Gate  
**Purpose:** Build the pure-Python reference implementations of (a) readiness predicate clauses, (b) auth-state classification with mocked providers, (c) token-store policy checks, (d) action registry + permission matrix, (e) artifact-layout / no-source-repo checks. These prototypes are the thing P4-P6 will lift into runtime modules.
**Output:** `configs/generated/standalone_graph_runtime_integration_pack_p3_prototypes_v1_verification.json`.

In [ ]:
import json, re, hashlib, os
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, Optional
REPO = Path('/Users/smarthaus/Projects/GitHub/M365').resolve()


## P3A - Notebook headers (purpose, lemmas, invariants, expected results)
Mapped to L98 lemmas. Expected: every prototype clause returns a deterministic boolean and every state classification falls inside the failure lattice.

In [ ]:
LATTICE = ('success','not_configured','auth_required','consent_required','permission_missing','throttled','graph_unreachable','policy_denied','mutation_fence','internal_error')
ALLOWED_AUTH = frozenset({'auth_code_pkce','device_code','app_only_secret','app_only_certificate'})
REDACT = re.compile(r'(?i)token|secret|password|assertion|certificate.*key|private.*key|authorization')


## P3B - Pure readiness and state-classification functions

In [ ]:
@dataclass
class Health:
    svc: Optional[bool] = None
    auth: Optional[bool] = None
    tok: Optional[bool] = None
    graph: Optional[bool] = None
    perm: Optional[bool] = None
    ctr: Optional[bool] = None
    art: Optional[bool] = None
    src: Optional[bool] = None
    aud: Optional[bool] = None

def readiness(h: Health):
    vals = (h.svc, h.auth, h.tok, h.graph, h.perm, h.ctr, h.art, h.src, h.aud)
    if any(v is None for v in vals): return ('not_ready','unknown_clause')
    if all(v is True for v in vals): return ('ready','success')
    failing = [name for name, v in zip(['svc','auth','tok','graph','perm','ctr','art','src','aud'], vals) if v is False]
    return ('not_ready', failing[0])

tests = [
    (Health(True,True,True,True,True,True,True,True,True), ('ready','success')),
    (Health(True,False,True,True,True,True,True,True,True), ('not_ready','auth')),
    (Health(True,True,True,None,True,True,True,True,True), ('not_ready','unknown_clause')),
    (Health(False,True,True,True,True,True,True,True,True), ('not_ready','svc')),
]
for h, expected in tests:
    assert readiness(h) == expected, (h, expected, readiness(h))
True

## P3C - Auth-state classification with mocked providers

In [ ]:
@dataclass
class AuthMaterial:
    has_token: bool
    expired: bool
    revoked: bool
    consent_pending: bool
    auth_mode: str

def classify_auth(m: AuthMaterial):
    if m.auth_mode not in ALLOWED_AUTH: return 'auth_required'
    if m.consent_pending: return 'consent_required'
    if m.revoked or not m.has_token: return 'auth_required'
    if m.expired: return 'auth_required'
    return 'signed_in'

auth_tests = [
    (AuthMaterial(True, False, False, False, 'auth_code_pkce'), 'signed_in'),
    (AuthMaterial(True, True, False, False, 'auth_code_pkce'), 'auth_required'),
    (AuthMaterial(False, False, False, False, 'app_only_secret'), 'auth_required'),
    (AuthMaterial(True, False, True, False, 'app_only_certificate'), 'auth_required'),
    (AuthMaterial(True, False, False, True, 'auth_code_pkce'), 'consent_required'),
    (AuthMaterial(True, False, False, False, 'password'), 'auth_required'),
]
for m, expected in auth_tests:
    assert classify_auth(m) == expected, (m, expected, classify_auth(m))
True

## P3C continued - Token-store policy checks

In [ ]:
def token_store_policy(store_kind: str, encrypted_at_rest: bool, allow_pack_local_encrypted: bool):
    if store_kind == 'keychain': return 'safe'
    if store_kind == 'encrypted_pack_local' and encrypted_at_rest and allow_pack_local_encrypted: return 'safe'
    if store_kind in {'plaintext_file','memory_only','env_vars','none'}: return 'unsafe'
    return 'unsafe'

store_tests = [
    ('keychain', False, False, 'safe'),
    ('encrypted_pack_local', True, True, 'safe'),
    ('encrypted_pack_local', False, True, 'unsafe'),
    ('encrypted_pack_local', True, False, 'unsafe'),
    ('plaintext_file', True, True, 'unsafe'),
    ('memory_only', True, True, 'unsafe'),
    ('env_vars', True, True, 'unsafe'),
]
for kind, enc, allow, expected in store_tests:
    assert token_store_policy(kind, enc, allow) == expected, (kind, enc, allow, expected)
True

## P3D - Action registry and permission matrix prototype

In [ ]:
READ_ONLY_REGISTRY = {
    'graph.org_profile':       {'workload':'directory','endpoint':'/organization','auth_modes':['app_only_secret','app_only_certificate'],'scopes':['Organization.Read.All'],'risk':'low','rw':'read'},
    'graph.me':                {'workload':'me','endpoint':'/me','auth_modes':['auth_code_pkce','device_code'],'scopes':['User.Read'],'risk':'low','rw':'read'},
    'graph.users.list':        {'workload':'users','endpoint':'/users','auth_modes':['auth_code_pkce','device_code','app_only_secret','app_only_certificate'],'scopes':['User.Read.All'],'risk':'low','rw':'read'},
    'graph.groups.list':       {'workload':'groups','endpoint':'/groups','auth_modes':['app_only_secret','app_only_certificate','auth_code_pkce','device_code'],'scopes':['Group.Read.All'],'risk':'low','rw':'read'},
    'graph.sites.root':        {'workload':'sharepoint','endpoint':'/sites/root','auth_modes':['app_only_secret','app_only_certificate','auth_code_pkce'],'scopes':['Sites.Read.All'],'risk':'low','rw':'read'},
    'graph.sites.search':      {'workload':'sharepoint','endpoint':'/sites?search=','auth_modes':['app_only_secret','app_only_certificate','auth_code_pkce'],'scopes':['Sites.Read.All'],'risk':'low','rw':'read'},
    'graph.teams.list':        {'workload':'teams','endpoint':'/teams','auth_modes':['app_only_secret','app_only_certificate','auth_code_pkce'],'scopes':['Team.ReadBasic.All'],'risk':'low','rw':'read'},
    'graph.drives.list':       {'workload':'sharepoint','endpoint':'/drives','auth_modes':['app_only_secret','app_only_certificate','auth_code_pkce'],'scopes':['Files.Read.All'],'risk':'low','rw':'read'},
    'graph.mail.health':       {'workload':'exchange','endpoint':'/me/mailFolders','auth_modes':['auth_code_pkce','device_code'],'scopes':['Mail.Read'],'risk':'low','rw':'read'},
    'graph.calendar.health':   {'workload':'exchange','endpoint':'/me/calendars','auth_modes':['auth_code_pkce','device_code'],'scopes':['Calendars.Read'],'risk':'low','rw':'read'},
    'graph.servicehealth':     {'workload':'admin','endpoint':'/admin/serviceHealth/issues','auth_modes':['app_only_secret','app_only_certificate'],'scopes':['ServiceHealth.Read.All'],'risk':'low','rw':'read'},
}
for aid, spec in READ_ONLY_REGISTRY.items():
    assert spec['rw'] == 'read'
    assert spec['scopes']
    assert all(am in ALLOWED_AUTH for am in spec['auth_modes'])
len(READ_ONLY_REGISTRY)

In [ ]:
def evaluate_permission(action_id, granted_scopes, current_auth_mode):
    spec = READ_ONLY_REGISTRY.get(action_id)
    if spec is None: return ('denied','unknown_action')
    if current_auth_mode not in spec['auth_modes']: return ('denied','auth_mode_mismatch')
    if not set(spec['scopes']).issubset(granted_scopes): return ('denied','permission_missing')
    return ('admit','ok')

perm_tests = [
    ('graph.me', {'User.Read'}, 'auth_code_pkce', ('admit','ok')),
    ('graph.me', set(), 'auth_code_pkce', ('denied','permission_missing')),
    ('graph.me', {'User.Read'}, 'app_only_secret', ('denied','auth_mode_mismatch')),
    ('does.not.exist', {'X'}, 'auth_code_pkce', ('denied','unknown_action')),
    ('graph.servicehealth', {'ServiceHealth.Read.All'}, 'app_only_secret', ('admit','ok')),
]
for aid, scopes, mode, expected in perm_tests:
    assert evaluate_permission(aid, scopes, mode) == expected, (aid, scopes, mode, expected)
True

## P3E - Artifact layout and no-source-repo checks

In [ ]:
EXPECTED_INSTALLED_LAYOUT = (
    'manifest.json',
    'payload.tar.gz',
    'signatures/manifest.sig',
    'signatures/payload.sig',
    'evidence/conformance.json',
    'SHA256SUMS',
)
EXPECTED_PAYLOAD_LAYOUT_TARGET = (
    'manifest.json',
    'setup_schema.json',
    'registry/agents.yaml',
    'registry/action_registry.yaml',
    'ucp_m365_pack/__init__.py',
    'ucp_m365_pack/contracts.py',
    'ucp_m365_pack/client.py',
    'm365_runtime/__init__.py',
    'm365_runtime/__main__.py',
    'm365_runtime/launcher.py',
    'm365_runtime/auth/__init__.py',
    'm365_runtime/auth/oauth.py',
    'm365_runtime/auth/app_only.py',
    'm365_runtime/auth/token_store.py',
    'm365_runtime/graph/__init__.py',
    'm365_runtime/graph/client.py',
    'm365_runtime/graph/errors.py',
    'm365_runtime/graph/registry.py',
    'm365_runtime/health.py',
    'm365_runtime/audit.py',
)
FORBIDDEN_TOKENS = ('M365_REPO_ROOT','SMARTHAUS_M365_REPO_ROOT','UCP_ROOT','REPOS_ROOT','UCP_REPOS_ROOT','../M365','from ops_adapter')

def installed_layout_ok(installed_files: Iterable[str]):
    files = set(installed_files)
    return set(EXPECTED_INSTALLED_LAYOUT).issubset(files)

def payload_layout_target_ok(payload_files: Iterable[str]):
    files = set(payload_files)
    return set(EXPECTED_PAYLOAD_LAYOUT_TARGET).issubset(files)

def no_source_repo_dependency_ok(file_text_iter):
    return all(not any(t in text for t in FORBIDDEN_TOKENS) for text in file_text_iter)

assert installed_layout_ok(EXPECTED_INSTALLED_LAYOUT)
assert payload_layout_target_ok(EXPECTED_PAYLOAD_LAYOUT_TARGET)
assert no_source_repo_dependency_ok(['def launch(): ...'])
assert not no_source_repo_dependency_ok(['if M365_REPO_ROOT: ...'])
assert not no_source_repo_dependency_ok(['from ops_adapter.main import app'])
True

## P3F - Scorecard emission

In [ ]:
verification = {
  'plan_ref': 'plan:m365-standalone-graph-runtime-integration-pack:R4',
  'phase': 'P3',
  'lattice': list(LATTICE),
  'allowed_auth': sorted(ALLOWED_AUTH),
  'readiness_test_count': 4,
  'auth_classification_test_count': 6,
  'token_store_test_count': 7,
  'action_registry_size': len(READ_ONLY_REGISTRY),
  'permission_test_count': 5,
  'expected_installed_layout': list(EXPECTED_INSTALLED_LAYOUT),
  'expected_payload_layout_target': list(EXPECTED_PAYLOAD_LAYOUT_TARGET),
  'forbidden_source_repo_tokens': list(FORBIDDEN_TOKENS),
  'truthful': True,
}
out = REPO / 'configs/generated/standalone_graph_runtime_integration_pack_p3_prototypes_v1_verification.json'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(verification, indent=2, sort_keys=True))
verification['truthful'], verification['action_registry_size']